In [0]:
"""
Bronze ingestion: Citi Bike trip data.

Schema drift decision:

Citi Bike changed its column schema starting with the 2020 01 data:

Legacy era (2014 01 to 2019 12):
Columns: tripduration, starttime, stoptime, start station id, start station name,
start station latitude, start station longitude, end station id,
end station name, end station latitude, end station longitude,
bikeid, usertype, birth year, gender

Current era (2020 01 to 2026 05 (present)):
Columns: ride_id, rideable_type, started_at, ended_at, start_station_name,
start_station_id, end_station_name, end_station_id, start_lat, start_lng,
end_lat, end_lng, member_casual

Create two separate bronze tables, one for each schema.
 - bronze.trips_raw_legacy: legacy schema
 - bronze.trips_raw_current: current schema

Each table's columns are what that era's CSVs contain (as strings, no inferSchema), plus the following columns on both tables: ingestion_timestamp, source_file, schema_era.

Consolidating the schema drift and differing columns will handled in the silver layer.
"""

In [0]:
from datetime import datetime, timezone

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

In [0]:
CATALOG = "citibike"
LANDING_SCHEMA = "landing"
LANDING_VOLUME = "raw"
LANDING_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{LANDING_VOLUME}"

BRONZE_SCHEMA = "bronze"
LEGACY_TABLE_PATH = f"{CATALOG}.{BRONZE_SCHEMA}.trips_raw_legacy"
CURRENT_TABLE_PATH = f"{CATALOG}.{BRONZE_SCHEMA}.trips_raw_current"

LEGACY_YEARS = [2014, 2015, 2016, 2017, 2018, 2019]
CURRENT_YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

LEGACY_GLOB = f"{LANDING_PATH}/{{{','.join(str(y) for y in LEGACY_YEARS)}}}-citibike-tripdata/*/*.csv"
CURRENT_GLOB = f"{LANDING_PATH}/{{{','.join(str(y) for y in CURRENT_YEARS)}}}-citibike-tripdata/*/*.csv"

In [0]:
# Type as all strings to prevent data quality errors raised
LEGACY_SCHEMA = StructType([
    StructField("tripduration", StringType(), True),
    StructField("starttime", StringType(), True),
    StructField("stoptime", StringType(), True),
    StructField("start station id", StringType(), True),
    StructField("start station name", StringType(), True),
    StructField("start station latitude", StringType(), True),
    StructField("start station longitude", StringType(), True),
    StructField("end station id", StringType(), True),
    StructField("end station name", StringType(), True),
    StructField("end station latitude", StringType(), True),
    StructField("end station longitude", StringType(), True),
    StructField("bikeid", StringType(), True),
    StructField("usertype", StringType(), True),
    StructField("birth year", StringType(), True),
    StructField("gender", StringType(), True),
])

CURRENT_SCHEMA = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", StringType(), True),
    StructField("ended_at", StringType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", StringType(), True),
    StructField("start_lng", StringType(), True),
    StructField("end_lat", StringType(), True),
    StructField("end_lng", StringType(), True),
    StructField("member_casual", StringType(), True),
])

In [0]:
def read_bronze_source(spark: SparkSession, glob_path: str, schema: StructType, era_label: str) -> DataFrame:
    """
    Read a set of CSVs with an defined schema and attach
    extra columns: source_file, ingestion_timestamp, schema_era.
    """
    df = (
        spark.read.format("csv")
        .schema(schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .load(glob_path)
        .withColumn("source_file", F.col("_metadata.file_path")) # F.input_file_name()
        .withColumn("ingestion_timestamp", F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn("schema_era", F.lit(era_label))
    )
    return df

In [0]:
def write_bronze(df: DataFrame, table_fqn: str, mode: str = "append") -> None:
    """
    Write to a bronze Delta table, creating the schema if needed.
    """
    spark = df.sparkSession
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

    (
        df.write.format("delta")
        .mode(mode)
        .option("mergeSchema", "true")
        
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .saveAsTable(table_fqn)
    )


In [0]:
spark = SparkSession.builder.getOrCreate()

print(f"Reading legacy era files from: {LEGACY_GLOB}")
legacy_df = read_bronze_source(spark, LEGACY_GLOB, LEGACY_SCHEMA, "legacy")
legacy_count = legacy_df.count()
print(f"{legacy_count} legacy rows")

print(f"Reading current era files from: {CURRENT_GLOB}")
current_df = read_bronze_source(spark, CURRENT_GLOB, CURRENT_SCHEMA, "current")
current_count = current_df.count()
print(f"{current_count} current rows")

print(f"Writing {LEGACY_TABLE_PATH} ...")
write_bronze(legacy_df, LEGACY_TABLE_PATH, mode="append")

print(f"Writing {CURRENT_TABLE_PATH} ...")
write_bronze(current_df, CURRENT_TABLE_PATH, mode="append")


Double check to make sure all the header columns in the legacy and current era are in order since Spark assumes headers are in order

In [0]:
def normalize_header(header_line):
    return [c.strip().strip('"').lower().replace(" ", "").replace("_", "") for c in header_line.split(",")]


def check_era(glob_path, schema, era_label):
    expected_normalized = [
        f.name.lower().replace(" ", "").replace("_", "") for f in schema.fields
    ]

    files = (
        spark.read.format("binaryFile")
        .load(glob_path)
        .select("path")
        .collect()
    )
    files = [row["path"] for row in files]

    mismatches = []
    for path in files:
        header_line = spark.read.text(path).limit(1).collect()[0]["value"]
        actual_normalized = normalize_header(header_line)

        if actual_normalized != expected_normalized:
            mismatches.append({"path": path, "header": header_line})

    print(f"[{era_label}] {len(files)} files checked, {len(mismatches)} mismatches found")
    for m in mismatches:
        print(f"  {m['path']}")
        print(f"    {m['header']}")

    return mismatches


legacy_mismatches = check_era(LEGACY_GLOB, LEGACY_SCHEMA, "legacy")
print()
current_mismatches = check_era(CURRENT_GLOB, CURRENT_SCHEMA, "current")